# 🗄️ Clase 2: Bases de Datos y Limpieza de Datos

**Curso: Modelos de AI en Negocios (MAIN) - IA Aplicada a la Industria** — Universidad de Santiago de Chile

**Autor:** [Byron Caices](https://www.linkedin.com/in/byron-caices/)

---

## 🏭 Contexto del caso

Somos analistas de datos de una **planta industrial de manufactura**. El equipo de producción nos ha entregado un archivo CSV con 200 registros diarios que incluyen información sobre turnos, máquinas, unidades producidas y defectos detectados.

**El problema:** los datos tienen errores reales — valores faltantes, tipos incorrectos y registros imposibles — que impiden usarlos directamente para tomar decisiones.

**Nuestra misión al terminar este notebook:**
1. Detectar y corregir los problemas de calidad del dataset
2. Calcular la **tasa de defectos** por turno y por máquina
3. Identificar **qué turno y qué máquina** presentan más problemas
4. Exportar un CSV limpio y generar gráficos listos para presentar a gerencia

---

## 📋 Diccionario de datos (columnas del CSV)

| Columna | Descripción | Tipo esperado |
|---------|-------------|---------------|
| `fecha` | Fecha del registro de producción | Fecha (guardada como texto — dato sucio) |
| `turno` | Turno de trabajo: Mañana, Tarde o Noche | Texto categórico |
| `maquina` | Máquina utilizada: A, B, C o D | Texto categórico |
| `tipo_defecto` | Tipo de falla detectada (puede ser nulo si no hubo defecto) | Texto / NaN |
| `unidades_producidas` | Número de unidades fabricadas en ese turno | Entero |
| `unidades_defectuosas` | Número de unidades con algún defecto | Entero |
| `operador` | Nombre del operador responsable (puede no estar registrado) | Texto / NaN |

---

## ⚠️ Problemas conocidos en los datos (lo que vamos a solucionar)

- **Valores nulos** en `tipo_defecto` (~30% de registros) y `operador` (~10%)
- **Fechas como texto** en lugar de tipo fecha/datetime
- **Registros imposibles**: algunos tienen `unidades_defectuosas > unidades_producidas` (error de ingreso manual)

---

**¿Cómo usar este cuaderno?**
- Ejecuten cada celda en orden con **Shift + Enter**
- Lean los comentarios y el texto antes de ejecutar — el objetivo es **entender** cada paso, no solo ejecutarlo

In [ ]:
# Importar las librerías necesarias
import numpy as np           # Cálculos numéricos eficientes
import pandas as pd          # Manejo de tablas de datos (DataFrames)
import matplotlib.pyplot as plt  # Generación de gráficos

print('✓ Librerías cargadas correctamente')
print('  - numpy: para operaciones matemáticas sobre arreglos')
print('  - pandas: para leer, limpiar y transformar datos en tablas')
print('  - matplotlib: para crear gráficos y visualizaciones')

---
## 📂 Paso 0: Cargar los datos

Normalmente en un proyecto real, cargaríamos el CSV descargado de Kaggle o de un sistema ERP:

```python
# Para cargar el CSV real de Kaggle (descomentar cuando tengan el archivo):
# df = pd.read_csv('defectos_manufactura.csv')
```

Para esta sesión, **generaremos el dataset directamente en Python** con las mismas columnas y los mismos tipos de problemas que tendría un archivo real de manufactura. Esto nos permite trabajar sin depender de una descarga.

> 💡 El proceso de limpieza que haremos es **exactamente el mismo** ya sea con datos sintéticos o con el CSV real.

In [ ]:
# ─── Generación del dataset sintético de manufactura ───
# Simulamos los datos que encontraríamos en un CSV real de una planta industrial

np.random.seed(42)  # Semilla fija para que todos obtengan los mismos resultados

n = 200  # Número de registros (200 días de producción)

# Generar datos base
unidades_prod = np.random.randint(80, 201, n)
unidades_def  = np.random.randint(0, 21, n)

# ──────────────────────────────────────────────────────
# DATO SUCIO 1: 5 registros imposibles (defectuosas > producidas)
# Simula errores de digitación al registrar los datos
idx_error = np.random.choice(n, size=5, replace=False)
unidades_def[idx_error] = unidades_prod[idx_error] + np.random.randint(1, 10, 5)
# ──────────────────────────────────────────────────────

# tipo_defecto: NaN en ~30% (cuando no hubo defecto, el campo queda vacío)
tipos_opciones = ['Dimensional', 'Superficial', 'Funcional', np.nan]
tipos_probs    = [0.30, 0.20, 0.20, 0.30]
tipo_defecto   = np.random.choice(tipos_opciones, size=n, p=tipos_probs)

# operador: NaN en ~10% (no siempre se registra quién operó)
operadores_opciones = ['García', 'Martínez', 'López', 'Rodríguez', 'Fernández', np.nan]
operadores_probs    = [0.20, 0.20, 0.20, 0.15, 0.15, 0.10]
operador = np.random.choice(operadores_opciones, size=n, p=operadores_probs)

# Fechas: guardadas como texto (DATO SUCIO 2)
fechas_datetime = pd.date_range(start='2024-01-02', periods=n, freq='D')
fechas_texto    = [str(f.date()) for f in fechas_datetime]  # Se guarda como string

# Crear el DataFrame
df = pd.DataFrame({
    'fecha':               fechas_texto,
    'turno':               np.random.choice(['Mañana', 'Tarde', 'Noche'], size=n),
    'maquina':             np.random.choice(['Máquina A', 'Máquina B', 'Máquina C', 'Máquina D'], size=n),
    'tipo_defecto':        tipo_defecto,
    'unidades_producidas': unidades_prod,
    'unidades_defectuosas':unidades_def,
    'operador':            operador,
})

print(f'✓ Dataset generado: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'  Rango de fechas: {df["fecha"].iloc[0]} → {df["fecha"].iloc[-1]}')

---
## 🔍 Paso 1: Exploración inicial — ¿Qué tenemos?

Antes de limpiar cualquier cosa, necesitamos **entender la estructura del dataset**. Esto es equivalente a abrir el archivo Excel por primera vez y revisar qué columnas tiene, cuántas filas hay, y si los tipos de datos tienen sentido.

Las preguntas clave son:
- ¿Cuántas filas y columnas tiene?
- ¿Qué tipo de dato tiene cada columna?
- ¿Cómo se ven los primeros registros?

In [ ]:
# Ver las primeras 5 filas del dataset
print('=== Primeras 5 filas del dataset ===')
df.head()

In [ ]:
# Información general: tipos de datos, cantidad de valores no nulos
print('=== Información general del dataset ===')
print(f'Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
print()
df.info()

In [ ]:
# Estadísticas descriptivas para las columnas numéricas
print('=== Estadísticas descriptivas (columnas numéricas) ===')
df.describe()

---
## 🚨 Paso 2: Detectar problemas de calidad

Ahora que conocemos la estructura, vamos a **identificar los problemas** de manera sistemática.

Un dato de mala calidad puede ser:
- **Nulo / vacío**: el campo no tiene valor
- **Tipo incorrecto**: una fecha guardada como texto, o un número guardado como texto
- **Valor imposible**: `unidades_defectuosas > unidades_producidas` no tiene sentido físico
- **Inconsistencia**: el mismo dato escrito de formas distintas (ej: "mañana" vs "Mañana")

In [ ]:
# Contar valores nulos por columna
nulos = df.isnull().sum()
porcentaje_nulos = (nulos / len(df) * 100).round(1)

resumen_nulos = pd.DataFrame({
    'Valores nulos': nulos,
    'Porcentaje (%)': porcentaje_nulos
})

print('=== Valores nulos por columna ===')
print(resumen_nulos)
print()
print(f'Total de celdas vacías: {nulos.sum()}')

In [ ]:
# Detectar registros imposibles: unidades_defectuosas > unidades_producidas
registros_imposibles = df[df['unidades_defectuosas'] > df['unidades_producidas']]

print(f'=== Registros con error (defectuosas > producidas): {len(registros_imposibles)} encontrados ===')
print(registros_imposibles[['fecha', 'turno', 'maquina', 'unidades_producidas', 'unidades_defectuosas']])

In [ ]:
# Verificar si 'fecha' es texto o fecha real
print(f'Tipo de dato de la columna fecha: {df["fecha"].dtype}')
print(f'Ejemplo de valor: {df["fecha"].iloc[0]} (tipo: {type(df["fecha"].iloc[0]).__name__})')
print()
print('→ Está guardada como texto (object/string), no como fecha. Hay que convertirla.')

---
## 🧹 Paso 3: Limpieza de datos

Ahora que sabemos **qué está mal**, vamos a corregirlo sistemáticamente. Este es el corazón del trabajo de un analista de datos.

El orden importa: primero convertimos tipos, luego tratamos nulos, y finalmente eliminamos registros inválidos.

In [ ]:
# Hacer una copia para no modificar el original (buena práctica)
df_limpio = df.copy()

# ─── CORRECCIÓN 1: Convertir 'fecha' de texto a tipo fecha ───
df_limpio['fecha'] = pd.to_datetime(df_limpio['fecha'])

print(f'Tipo ANTES: {df["fecha"].dtype}')
print(f'Tipo DESPUÉS: {df_limpio["fecha"].dtype}')
print(f'Ejemplo convertido: {df_limpio["fecha"].iloc[0]}')

In [ ]:
# ─── CORRECCIÓN 2: Tratar valores nulos ───

# tipo_defecto nulo → significa que NO hubo defecto ese día/turno
# Es correcto reemplazarlo con 'Sin defecto'
df_limpio['tipo_defecto'] = df_limpio['tipo_defecto'].fillna('Sin defecto')

# operador nulo → el dato simplemente no fue registrado
# Lo marcamos como 'No registrado' para distinguirlo de un valor real
df_limpio['operador'] = df_limpio['operador'].fillna('No registrado')

# Verificar que ya no hay nulos
nulos_restantes = df_limpio.isnull().sum().sum()
print(f'Valores nulos ANTES: {df.isnull().sum().sum()}')
print(f'Valores nulos DESPUÉS: {nulos_restantes}')

In [ ]:
# ─── CORRECCIÓN 3: Eliminar registros imposibles ───

filas_antes = len(df_limpio)

# Conservar solo registros donde defectuosas <= producidas
df_limpio = df_limpio[df_limpio['unidades_defectuosas'] <= df_limpio['unidades_producidas']]

filas_despues = len(df_limpio)
eliminados = filas_antes - filas_despues

print(f'Filas ANTES de limpiar: {filas_antes}')
print(f'Filas DESPUÉS de limpiar: {filas_despues}')
print(f'Registros eliminados (imposibles): {eliminados}')

In [ ]:
# Resumen del dataset limpio
print('=== Estado final del dataset limpio ===')
print(f'Dimensiones: {df_limpio.shape[0]} filas × {df_limpio.shape[1]} columnas')
print()
print('Tipos de datos:')
print(df_limpio.dtypes)
print()
print('Valores nulos:')
print(df_limpio.isnull().sum())

---
## ⚙️ Paso 4: Calcular nuevas métricas de negocio

Con los datos limpios, ahora podemos calcular **indicadores que no existían en el dataset original** pero que son clave para tomar decisiones.

La métrica principal que nos interesa es la **tasa de defectos**: ¿qué porcentaje de lo que producimos sale con falla?

$$\text{Tasa de defectos} = \frac{\text{Unidades defectuosas}}{\text{Unidades producidas}} \times 100$$

In [ ]:
# Calcular tasa de defectos por registro
df_limpio['tasa_defectos'] = (
    df_limpio['unidades_defectuosas'] / df_limpio['unidades_producidas'] * 100
).round(2)

print('=== Dataset con nueva columna tasa_defectos ===')
df_limpio[['fecha', 'turno', 'maquina', 'unidades_producidas', 'unidades_defectuosas', 'tasa_defectos']].head(8)

In [ ]:
# ─── Análisis por turno ───
resumen_turno = df_limpio.groupby('turno').agg(
    total_producidas   = ('unidades_producidas', 'sum'),
    total_defectuosas  = ('unidades_defectuosas', 'sum'),
    tasa_promedio      = ('tasa_defectos', 'mean'),
    registros          = ('fecha', 'count')
).round(2).sort_values('tasa_promedio', ascending=False)

print('=== Resumen por turno ===')
print(resumen_turno)
print()
turno_problematico = resumen_turno.index[0]
print(f'→ El turno con mayor tasa de defectos es: {turno_problematico}')

In [ ]:
# ─── Análisis por máquina ───
resumen_maquina = df_limpio.groupby('maquina').agg(
    total_producidas   = ('unidades_producidas', 'sum'),
    total_defectuosas  = ('unidades_defectuosas', 'sum'),
    tasa_promedio      = ('tasa_defectos', 'mean'),
    registros          = ('fecha', 'count')
).round(2).sort_values('tasa_promedio', ascending=False)

print('=== Resumen por máquina ===')
print(resumen_maquina)
print()
maquina_problematica = resumen_maquina.index[0]
print(f'→ La máquina con mayor tasa de defectos es: {maquina_problematica}')

---
## 📊 Paso 5: Visualizaciones para toma de decisiones

Los números en tablas son útiles, pero un gráfico bien hecho comunica en segundos lo que una tabla no puede. Vamos a crear dos visualizaciones:

1. **Gráfico de barras**: tasa de defectos promedio por turno y por máquina
2. **Gráfico de línea**: evolución de la tasa de defectos en el tiempo

In [ ]:
# ─── Gráfico 1: Tasa de defectos por turno y por máquina ───

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# --- Panel izquierdo: por turno ---
turnos_orden   = resumen_turno.index.tolist()
tasas_turno    = resumen_turno['tasa_promedio'].tolist()
colores_turno  = ['#EA7600' if t == turno_problematico else '#00a499' for t in turnos_orden]

barras1 = ax1.bar(turnos_orden, tasas_turno, color=colores_turno, edgecolor='white', linewidth=1.5)
for barra, valor in zip(barras1, tasas_turno):
    ax1.text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 0.1,
             f'{valor:.1f}%', ha='center', fontweight='bold', fontsize=11)

ax1.set_title('Tasa de Defectos Promedio por Turno', fontsize=13, fontweight='bold')
ax1.set_ylabel('Tasa de defectos (%)')
ax1.set_ylim(0, max(tasas_turno) * 1.25)
ax1.grid(axis='y', alpha=0.3)

# --- Panel derecho: por máquina ---
maquinas_orden    = resumen_maquina.index.tolist()
tasas_maquina     = resumen_maquina['tasa_promedio'].tolist()
colores_maquina   = ['#EA7600' if m == maquina_problematica else '#00a499' for m in maquinas_orden]

barras2 = ax2.bar(maquinas_orden, tasas_maquina, color=colores_maquina, edgecolor='white', linewidth=1.5)
for barra, valor in zip(barras2, tasas_maquina):
    ax2.text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 0.1,
             f'{valor:.1f}%', ha='center', fontweight='bold', fontsize=11)

ax2.set_title('Tasa de Defectos Promedio por Máquina', fontsize=13, fontweight='bold')
ax2.set_ylabel('Tasa de defectos (%)')
ax2.set_ylim(0, max(tasas_maquina) * 1.25)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Análisis de Calidad — Planta de Manufactura', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Barra naranja = mayor tasa de defectos → punto de atención prioritario')

In [ ]:
# ─── Gráfico 2: Evolución de la tasa de defectos en el tiempo ───
# Agrupamos por fecha para ver la tendencia diaria

evolucion = df_limpio.groupby('fecha')['tasa_defectos'].mean().reset_index()

# Promedio móvil de 7 días para suavizar la línea
evolucion['promedio_movil'] = evolucion['tasa_defectos'].rolling(window=7, min_periods=1).mean()

plt.figure(figsize=(13, 5))
plt.plot(evolucion['fecha'], evolucion['tasa_defectos'],
         color='#00a499', alpha=0.4, linewidth=1, label='Tasa diaria')
plt.plot(evolucion['fecha'], evolucion['promedio_movil'],
         color='#EA7600', linewidth=2.5, label='Promedio móvil (7 días)')

# Línea de referencia: tasa aceptable
plt.axhline(y=8, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Umbral de alerta (8%)')

plt.title('Evolución de la Tasa de Defectos en el Tiempo', fontsize=13, fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('Tasa de defectos (%)')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 💾 Paso 6: Exportar los datos limpios

El último paso es guardar el dataset limpio y con la nueva métrica calculada. Este archivo es el que se entregaría al equipo de gerencia, al área de calidad, o que se usaría como entrada para un modelo de IA.

In [ ]:
# Exportar el dataset limpio a CSV
nombre_archivo = 'defectos_manufactura_limpio.csv'
df_limpio.to_csv(nombre_archivo, index=False)

print(f'✓ Archivo exportado: {nombre_archivo}')
print(f'  Filas guardadas: {len(df_limpio)}')
print(f'  Columnas: {list(df_limpio.columns)}')
print()
print('Este archivo limpio está listo para:')
print('  • Entregarle al equipo de calidad con los gráficos')
print('  • Usarlo como entrada para un modelo de Machine Learning')
print('  • Importarlo en Power BI o Tableau para un dashboard')

---
## 🎯 Conclusión: ¿Qué aprendimos hoy?

### Respuesta a la pregunta de negocio

Con los datos limpios pudimos responder:
- **¿Qué turno tiene más defectos?** → Revisaron el output de la celda de análisis por turno
- **¿Qué máquina tiene más defectos?** → Revisaron el output de la celda de análisis por máquina
- **¿Hay una tendencia en el tiempo?** → El gráfico de evolución temporal muestra si empeora o mejora

Estas tres respuestas permiten a gerencia **priorizar dónde actuar** (inspeccionar la máquina más problemática, reforzar el turno con peor rendimiento, etc.).

### Lo que practicamos hoy

| Paso | Lo que hicimos | Herramienta |
|------|---------------|-------------|
| Cargar datos | `pd.read_csv()` / datos sintéticos | `pandas` |
| Explorar | `.info()`, `.describe()`, `.head()` | `pandas` |
| Detectar problemas | `.isnull().sum()`, filtros lógicos | `pandas` |
| Limpiar | `.fillna()`, `pd.to_datetime()`, filtros | `pandas` |
| Calcular métricas | Nueva columna, `.groupby()` | `pandas` |
| Visualizar | Barras, líneas, promedio móvil | `matplotlib` |
| Exportar | `.to_csv()` | `pandas` |

### Conexión con IA

> El trabajo que hicimos hoy — limpiar datos, calcular métricas, identificar patrones — es exactamente la **fase de preparación de datos** que precede a cualquier modelo de Machine Learning. Sin este paso, los modelos aprenden patrones incorrectos y producen resultados que no sirven para tomar decisiones.

**En la próxima sesión:** usaremos datos ya limpios para entrenar modelos de clasificación y predicción. 🚀